# Atcoder Typical DP Contest

> "전형적인 DP 문제를 풀어보자"

- toc: true
- branch: master
- badges: true
- comments: true
- author: 한재수
- categories: [Algorithm]

`-` 대회 링크: https://atcoder.jp/contests/tdpc

`-` 현재 [EDPC](https://atcoder.jp/contests/dp)에서 풀지 못한 문제가 5개(U, V, W, X, Y) 남았는데 오래 걸릴 것 같아서 여기로 이사왔다 

`-` 총 20 문제가 있는데 시간 날 때마다 풀어보자 (일본어라 번역해야 됨). 이번엔 다 풀 수 있을까?

## A - Contest

`-` $N$개의 질문을 고려하자. 각 질문을 해결하여 점수를 얻거나 해결하지 않아 점수를 얻지 않을 수 있다. 이를 통해 최종 점수로 가능한 것들이 총 몇 개 있는지 세면 된다

`-` $\operatorname{dp}[i][s]$를 $i$번째 질문까지 고려했을 때 $s$점을 획득할 수 있으면 $\text{True}$, 아니면 $\text{False}$로 정의하자. 그럼 점화식은 다음과 같다

`-` $\operatorname{dp}[i][s] = \operatorname{dp}[i - 1][s] \lor \operatorname{dp}[i - 1][s - p_i]$. base case로 $\operatorname{dp}[0][0] = \text{True},\, \operatorname{dp}[i][p_i] = \text{True}$이며 정답은 $\sum \operatorname{dp}[N]$이다

`-` 문제 조건에서 $p_i \le 100$이므로 최대 획득 점수는 $100N$에 바운드된다. 따라서 전체 알고리즘의 시간 복잡도는 $O\left(100N^2\right)$이다

`-` 배낭 용량과 물건 가치는 고려하지 않고 만들 수 있는 배낭 무게의 가짓수만 관심있는 배낭 문제라고 생각할 수 있겠다~

In [7]:
#collapse
def solution():
    N = int(input())
    array = list(map(int, input().split()))
    max_score = 100 * N
    dp = [0] * (max_score + 1)
    dp[0] = 1 
    for i in range(N):
        p_i = array[i]
        for s in range(max_score, p_i, -1):
            dp[s] |= dp[s - p_i]
        dp[p_i] = 1
    answer = sum(dp)
    print(answer)


solution()

`-` 배열로 풀긴 했지만 이런 유형은 해시셋 딸깍 하는 게 더 쉽다

`-` 비슷한 문제로 [양팔저울](https://jungol.co.kr/problem/1352)이 있다

## B - Game

`-` 각자 번갈아가며 왼쪽 더미와 오른쪽 더미 중 하나를 골라 맨 위의 물건을 가져간다. 각자가 최선을 다했을 때 선공 플레이어가 가져간 물건의 총 가치를 구하면 된다

`-` 최종적으로 선공 플레이어가 가져간 물건의 총 가치와 후공 플레이어가 가져간 물건의 총 가치의 합은 전체 물건의 총 가치와 동일하다. 최선을 다하는 건 게임에서 승리하도록 행동한다는 것이고 이는 내가 가져간 물건의 총 가치가 상대가 가져간 물건의 총 가치보다 크게 만드는 것이다. 각 플레이어는 지금 점수(=가져간 물건의 총 가치)를 많이 획득하는 것에 신경을 쓰는 게 아니라 게임이 종료되는 시점에 점수 차이가 극대화되는 선택을 한다

`-` $\operatorname{dp}[i][j]$를 밑에서부터 왼쪽 더미의 물건 $i$개와 오른쪽 더미의 물건 $j$개만 고려한 게임에서 각자 최선을 다해 플레이 할 때 게임 종료시 $\text{본인의 점수 $-$ 상대방의 점수}$로 정의하자. 현재 플레이어는 왼쪽 더미에서 물건을 가져가거나 오른쪽 더미에서 물건을 가져갈 수 있다. 따라서 점화식은 다음과 같다

`-` $\operatorname{dp}[i][j] = \max(a_{A-i + 1} - \operatorname{dp}[i - 1][j],\, b_{B-j+1} - \operatorname{dp}[i][j - 1])$

`-` base case로 $\operatorname{dp}[0][0] = 0$이다. 게임이 종료된 시점에서 $\text{선공 플레이어의 점수 $-$ 후공 플레이어의 점수 = $\operatorname{dp}[A][B]$}$이다

`-` $\text{선공 플레이어의 점수 $+$ 후공 플레이어의 점수 $=$ 물건의 총 가치}$이므로 연립 방정식을 통해 정답을 얻을 수 있다. 전체 알고리즘의 시간 복잡도는 $O(AB)$이다

`-` 바텁업으로 구현할 때 남은 물건 개수를 기준으로 오름차순으로 계산해야 함에 주의하자 (물건이 $1$개 남았을 때 가능한 경우 계산 $\to$ 물건이 $2$개 남았을 때 가능한 경우 계산 $\to \cdots \to$ 물건이 $A+B$개 남았을 때 가능한 경우 계산)

In [51]:
#collapse
def solution():
    inf = 10**6
    A, B = map(int, input().split())
    a = list(map(int, input().split()))
    b = list(map(int, input().split()))
    dp = [[0] * (B + 1) for _ in range(A + 1)]
    for remainder in range(1, A + B + 1):
        for i in range(max(remainder - B, 0), min(remainder, A) + 1):
            j = remainder - i
            left_pick = a[A - i] - dp[i - 1][j] if i > 0 else -inf
            right_pick = b[B - j] - dp[i][j - 1] if j > 0 else -inf
            dp[i][j] = max(left_pick, right_pick)
    answer = (dp[A][B] + sum(a) + sum(b)) // 2
    print(answer)


solution()

`-` 이런 유형을 Minimax DP로 부른다

`-` 비슷한 문제로 [Deque](https://atcoder.jp/contests/dp/tasks/dp_l)이 있다

## C - Tournament

`-` 사람별로 토너먼트 우승 확률을 구하는 문제이다

`-` 토너먼트는 트리 형태임에 주목하자. 임의의 노드 $u$에 대해 $\operatorname{dp}[u][i]$를 $i$번 사람이 노드 $u$의 서브트리 토너먼트에서 우승할 확률로 정의하자. 노드 $u$의 승자는 $u$의 왼쪽 자식 $l$의 승자와 오른쪽 자식 $r$의 승자 중 이긴 사람이 된다. 점화식은 다음과 같다

`-` $\operatorname{dp}[u][i] = \operatorname{dp}[l][i] \cdot \sum\limits_{j=1}^{2^K} \{\operatorname{dp}[r][j] \cdot f(i, j)\} + \operatorname{dp}[r][i]\cdot \sum\limits_{j=1}^{2^K}\{\operatorname{dp}[l][j] \cdot f(i, j)\}, \quad f(i,j) = 1 \mathbin{/} \left(1 + 10^{\frac{R_j - R_i}{400}}\right)$

`-` 노드 $u$에 $i$가 속하지 않는 경우 $\operatorname{dp}[u][i] = 0$임에 유의하자. base case로 리프 노드 $t$와 그 때의 사람 $i$에 대해 $\operatorname{dp}[u][i] = 1$이다

`-` 상태 공간의 크기는 $O\left(2^{2K}\right)$이고 전이가 $O\left(2^{K}\right)$이므로 총 $O\left(2^{3K}\right)$이다. $K$는 최대 $10$이므로 TLE이다. 실제로 필요한 경우만 저장하도록 수정하자

`-` $\operatorname{dp}[u]$를 해시맵으로 구현하고 실제 서브트리에 존재하는 사람만 저장하자. 트리의 깊이별로 사람의 수는 $O\left(2^K\right)$이다. 따라서 상태 공간의 크기는 $O\left(K 2^{K}\right)$이다

`-` 또한, 모든 가능한 매치 횟수는 $\sum\limits_{d=0}^{K - 1} 2^{d}\cdot  2^{2(K - d - 1)} = 2^{2(K-1)} \sum\limits_{d=0}^{K - 1} \frac{1}{2^{d}} = O\left(2^{2K}\right)$이다

`-` 따라서 전체 알고리즘의 시간 복잡도는 $O\left(2^{2K}\right)$이고 공간 복잡도는 $O\left(K2^{K}\right)$이다

`-` 완전 이진 트리에서 노드 $u$의 왼쪽 자식과 오른쪽 자식이 각각 $2u, 2u+1$임을 이용하면 바텀업으로 간단하게 구현할 수 있다

In [4]:
#collapse
from collections import defaultdict


def f(R_i, R_j):
    return 1 / (1 + 10**((R_j - R_i) / 400))


def solution():
    K = int(input())
    N = 2**K
    R = [int(input()) for _ in range(N)]
    dp = [defaultdict(int) for _ in range(2 * N)]
    for i, u in enumerate(range(N, 2 * N)):
        dp[u][i] = 1
    for u in range(N - 1, 0, -1):
        for i, p_i in dp[2 * u].items():
            for j, p_j in dp[2 * u + 1].items():
                dp[u][i] += p_i * p_j * f(R[i], R[j])
                dp[u][j] += p_i * p_j * f(R[j], R[i])
    for i in range(N):
        print(dp[1][i])


solution()

## D - Dice

`-` 주사위를 $N$번 던져 나온 눈의 곱이 $D$의 배수일 확률을 계산하면 된다

### MLE 풀이

`-` $\operatorname{dp}[i][r]$을 주사위를 $i$번 던져 나온 눈의 곱을 $D$로 나눈 나머지가 $r$일 확률로 정의하자. 주사위를 던지면 $1 \sim 6$ 중 하나가 균등확률로 나오므로 점화식은 다음과 같다

`-` $\forall x \in \{1,2,3,4,5,6\}, \quad \operatorname{dp}[i][(rx) \bmod D] \mathrel{+}= \frac{1}{6}\operatorname{dp}[i - 1][r]$

`-` 이때 $r$은 $0 \sim D - 1$이고 $D$는 최대 $10^{18}$이니 배열로 선언하면 MLE이다. 따라서 $\operatorname{dp}[i]$를 해시맵으로 선언해야 된다

`-` 문제는 상태 공간이 얼마나 희소하냐는 것이다. $1$은 곱해도 그대로라 이전 항이 그대로 누적되며 $2,3,4,5,6$을 곱하면 새로운 값이 되므로 이론상 $\operatorname{dp}[N]$의 크기는 $\sum\limits_{k=1}^{N} {_5 H_k} = O\left(N^5\right)$이다

`-` 전체 알고리즘의 시간 복잡도는 $O\left(N^6\right)$이고 $N$은 최대 $100$이므로 원칙적으로 TLE, MLE지만 계수가 작고 $4=2\times 2,\,6=2 \times 3$이므로 겹치는 상태가 분명히 존재하니 AC를 노려볼 수 있겠다

In [46]:
#collapse
from collections import defaultdict


def solution():
    N, D = map(int, input().split())
    Ω = list(range(1, 7))
    p = 1 / 6
    dp_prev = defaultdict(int)
    for x in Ω:
        dp_prev[x % D] += p
    for _ in range(N - 1):
        dp_curr = defaultdict(int)
        for r, f_r in dp_prev.items():
            for x in Ω:
                dp_curr[(r * x) % D] += p * f_r
        dp_prev = dp_curr
    answer = dp_prev[0]
    print(answer)


solution()

`-` 안타깝게도 테케 하나에서 MLE가 발생했다

### AC 풀이

`-` $1$과 $6$ 사이의 소인수는 $2,3,5$뿐이라 $D = 2^a 3^b 5^c$ 형태임에 주목하자. $a$는 최대 $2N$, $b$와 $c$는 최대 $N$으로 가능한 경우의 수는 $O\left(N^3\right)$이다

`-` $\operatorname{dp}[i][a][b][c]$를 주사위를 $i$번 던져 나온 눈의 곱이 $2^a \cdot 3^b \cdot 5^c$일 확률로 정의하자. 점화식은 다음과 같다

`-` $\operatorname{dp}[i][a][b][c] = \frac{1}{6}(\operatorname{dp}[i - 1][a][b][c] + \operatorname{dp}[i - 1][a - 1][b][c] + \operatorname{dp}[i - 1][a][b - 1][c] + \operatorname{dp}[i - 1][a - 2][b][c] + \operatorname{dp}[i - 1][a][b][c - 1] + \operatorname{dp}[i - 1][a - 1][b - 1][c])$

`-` base case로 $\operatorname{dp}[0][0][0][0]= 1$이며 정답은 $\sum\limits_{D \mid 2^a 3^b 5^c}\operatorname{dp}[N][a][b][c]$이다

`-` 전체 알고리즘의 시간 복잡도는 $O\left(N^4\right)$이지만 $\left\lfloor\frac{a + b}{2}\right\rfloor + c \le i,\, b \le i$를 만족해야 하므로 계수가 작아 TLE가 발생하지 않는다. 또한 토글링을 사용하면 공간 복잡도도 $O\left(N^3\right)$이므로 MLE가 발생하지 않아 AC를 받을 수 있다

In [101]:
#collapse
def init(n):
    return [[[0] * (n + 1) for _ in range(n + 1)] for _ in range(2 * n + 1)]


def solution():
    N, D = map(int, input().split())
    p = 1 / 6
    dp = [init(N), init(N)]
    dp[0][0][0][0] = 1
    for i in range(1, N + 1):
        dp_curr = dp[i % 2]
        dp_prev = dp[1 - i % 2]
        for a in range(2 * i + 1):
            for b in range(i + 1):
                for c in range(i - (a + b) // 2 + 1):
                    dp_curr[a][b][c] = dp_prev[a][b][c]
                    if a >= 1:
                        dp_curr[a][b][c] += dp_prev[a - 1][b][c]
                    if b >= 1:
                        dp_curr[a][b][c] += dp_prev[a][b - 1][c]
                    if c >= 1:
                        dp_curr[a][b][c] += dp_prev[a][b][c - 1]
                    if a >= 2:
                        dp_curr[a][b][c] += dp_prev[a - 2][b][c]
                    if a >= 1 and b >= 1:
                        dp_curr[a][b][c] += dp_prev[a - 1][b - 1][c]
                    dp_curr[a][b][c] *= p
    answer = 0
    for a in range(2 * N + 1):
        for b in range(N + 1):
            for c in range(N - (a + b) // 2 + 1):
                if (pow(2, a, D) * pow(3, b, D) * pow(5, c, D)) % D == 0:
                    answer += dp_curr[a][b][c]
    print(answer)


solution()

## E - Number

`-` $1\sim N$ 중에서 각 자릿수 합이 $D$의 배수인 수가 몇 개 있는지 세면 된다

`-` [Digit Sum](https://atcoder.jp/contests/dp/tasks/dp_s) 문제와 완전히 같아서 풀이는 생략하겠다 (대신 링크 남김)

`-` https://jaesu26.github.io/study-blog/algorithm/2026/08/05/atcoder-educational-dp-contest.html#S---Digit-Sum

## F - Semi-express

`-` 총 $N$개의 역이 있다. $1,N$번 역에선 정차해야 하며 연속해서 $K$번 이상 정차할 수 없을 때 열차가 정차할 수 있는 역들의 부분 집합이 몇 개인지 세면 된다

`-` $\operatorname{dp}[i][s]$를 열차가 현재 $i$번 역에 있고 연속해서 $s$번 정차한 상태로 가능한 경우의 수라 하자. 현재 연속으로 $s$번 정차한 상태라면 이전엔 무조건 연속으로 $s-1$번 정차한 상태여야 한다. 또한 연속으로 최대 $K-1$번까지 정차할 수 있으므로 점화식은 다음과 같다

`-` $\operatorname{dp}[i][s] = \begin{cases} 
\,  \sum\limits_{t=0}^{K-1}\operatorname{dp}[i - 1][t], & \text{if $s = 0$}  \\ \\
\,\operatorname{dp}[i - 1][s - 1], &  \text{if $0 < s < K$}
\end{cases}$

`-` base case로 $\operatorname{dp}[1][1] = 1$이고 정답은 $\sum\limits_{t=1}^{K-1}\operatorname{dp}[N][t]$이다. 근데 상태 공간의 크기가 $O(NK)$라 나이브하게 구현하면 TLE이다

`-` $s=0$을 제외하곤 $\operatorname{dp}[i][s] = \operatorname{dp}[i - 1][s - 1]$임에 주목하자. $\operatorname{dp}[i]$는 $\operatorname{dp}[i - 1]$에서 $\sum \operatorname{dp}[i - 1]$을 맨 앞에 추가하고 맨 뒤의 원소를 삭제한 것과 같다

`-` $\sum \operatorname{dp}[i] = 2\sum \operatorname{dp}[i - 1] - \operatorname{dp}[i - 1][K-1]$이므로 $O(1)$에 갱신할 수 있고 deque을 사용하면 appendleft와 pop도 $O(1)$에 가능하므로 전체 알고리즘의 시간 복잡도는 $O(N + K)$가 되며 공간 복잡도는 $O(K)$가 된다

In [13]:
#collapse
from collections import deque


def solution():
    mod = 10**9 + 7
    N, K = map(int, input().split())
    dp = deque([0] + [1] + [0] * (K - 2))
    Σ = 1
    for _ in range(N - 1):
        dp.appendleft(Σ)
        Σ = (2 * Σ - dp.pop()) % mod
    answer = (Σ - dp[0]) % mod
    print(answer)


solution()

`-` 더 어렵지만 deque를 사용해 최적화하는 문제로 [잔디밭 가꾸기](https://jungol.co.kr/problem/3621)가 있다

## H - knapsack

`-` 풀이 정리 예정

## J - Ball

`-` 각 위치 $0 \le x_i \le 15$에 대해 물건이 없거나 $1$개 있다. 위치 $x$에 공을 던지면 $x-1,x,x+1$ 중 한 곳에 $\frac{1}{3}$의 확률로 날아갈 때 모든 물건을 넘어뜨리기 위해 공을 던지는 횟수의 기댓값을 계산하자

`-` 물건끼리는 상태가 서로 독립이며 공이 날아가는 확률 또한 항상 같다. 따라서 가장 왼쪽의 물건이 위치 $x$에 있을 때 이를 넘어뜨리기 위해 $x-2,x-1$에 공을 던지는 건 손해이므로 왼쪽부터 그리디하게 처리하면 된다

`-` $f(s, p_1, p_2, p_3)$를 위치 $0 \sim s-1$에 있는 물건을 전부 넘어뜨렸고 $s, s+1, s+2$에 물건이 $p_1, p_2,p_3$개 존재할 때 모든 물건을 넘어뜨리기 위해 공을 던지는 횟수의 기댓값으로 정의하자. 각 상황마다 함숫값이 어떻게 변화하는지 살펴보자

`-` $s \ge 16$이면 모든 물건을 넘어뜨렸으므로 함숫값은 $0$이다

`-` $s < 16, p_1 = 0$이면 조준점을 $1$칸 오른쪽으로 옮기자. $f(s, p_1, p_2, p_3) = f(s + 1, p_2, p_3, k)$이며 $k$는 위치 $s + 3$에 존재하는 물건의 개수이다

`-` $p_1 \ne 0, p_2 = 0, p_3=0$이면 위치 $s$에 존재하는 물건만 넘어뜨리면 된다. 넘어뜨릴 확률은 $\frac{1}{3}$이므로 기댓값은 $3$이다. 따라서 $f(s, p_1, p_2, p_3) = 3p_1 + f(s, 0, 0, 0)$이다

`-` $p_1 \ne 0, p_2 = 0, p_3 \ne 0$이면 물건을 넘어뜨릴 확률은 $\frac{2}{3}$이므로 기댓값은 $\frac{3}{2}$이다. 위치 $s+1$에 공을 던져서 물건을 넘어뜨렸다면 $s, s+2$ 중 한 곳이며 각각 $\frac{1}{2}$이므로 $f(s, p_1, p_2, p_3) = \frac{3}{2} + \frac{1}{2}\{f(s, p_1 - 1, 0, p_3) + f(s, p_1, 0, p_3 - 1)\}$이다. $p_1 \ne 0, p_2 \ne 0, p_3 = 0$인 경우도 이와 같은 방식으로 계산하면 된다

`-` 마지막으로 $p_1 \ne 0, p_2 \ne 0, p_3 \ne 0$이면 물건을 넘어뜨릴 확률은 $1$이므로 기댓값도 $1$이다. $s, s+1, s+2$ 중 한 곳에 $\frac{1}{3}$의 확률로 날아갈테니 $f(s, p_1, p_2, p_3) = 1 + \frac{1}{3}\{f(s, p_1 - 1, p_2, p_3) + f(s, p_1 , p_2 - 1, p_3) + f(s, p_1, p_2, p_3 - 1)\}$이다

`-` 모든 상황을 정리한 점화식은 다음과 같다

$$f(s, p_1, p_2, p_3) = \begin{cases} 
\, 0 , & \text{if $s \ge 16$}  \\ \\
\,f(s + 1, p_2, p_3, k), &  \text{if $s < 16,p_1 = 0$}\\ \\
\,3p_1 + f(s, 0, 0, 0), &  \text{if $p_1 \ne 0, p_2 = 0, p_3=0$} \\ \\
\,\frac{3}{2} + \frac{1}{2}\{f(s, p_1 - 1, 0, p_3) + f(s, p_1, 0, p_3 - 1)\}, &  \text{if $p_1 \ne 0, p_2 = 0, p_3 \ne 0$} \\ \\
\,\frac{3}{2} + \frac{1}{2}\{f(s, p_1 - 1, p_2, 0) + f(s, p_1, p_2 - 1, 0)\}, &  \text{if $p_1 \ne 0, p_2 \ne 0, p_3 = 0$}\\ \\
\,1 + \frac{1}{3}\{f(s, p_1 - 1, p_2, p_3) + f(s, p_1 , p_2 - 1, p_3) + f(s, p_1, p_2, p_3 - 1)\}, &  \text{if $p_1 \ne 0, p_2 \ne 0, p_3 \ne 0$}
\end{cases}$$

`-` 가능한 상태 공간의 크기는 $O(16 \times 2 \times 2 \times 2)$이며 전이는 $O(1)$이므로 전체 알고리즘의 시간 복잡도는 $O\left(16 \times 2^3\right)$이다

In [39]:
#collapse
def f(s, p1, p2, p3, ks, max_x, dp):
    if s > max_x:
        return 0
    state = (s, p1, p2, p3)
    if state in dp:
        return dp[state]
    if p1 == 0:
        k = ks[s + 3] if s + 3 <= max_x else 0
        dp[state] = f(s + 1, p2, p3, k, ks, max_x, dp)
    elif p2 == 0 and p3 == 0:
        dp[state] = 3 * p1 + f(s, 0, 0, 0, ks, max_x, dp)
    elif p2 == 0:
        f1 = f(s, p1 - 1, 0, p3, ks, max_x, dp)
        f2 = f(s, p1, 0, p3 - 1, ks, max_x, dp)
        dp[state] = 1.5 + (f1 + f2) / 2
    elif p3 == 0:
        f1 = f(s, p1 - 1, p2, 0, ks, max_x, dp)
        f2 = f(s, p1, p2 - 1, 0, ks, max_x, dp)
        dp[state] = 1.5 + (f1 + f2) / 2
    else:
        f1 = f(s, p1 - 1, p2, p3, ks, max_x, dp)
        f2 = f(s, p1, p2 - 1, p3, ks, max_x, dp)
        f3 = f(s, p1, p2, p3 - 1, ks, max_x, dp)
        dp[state] = 1 + (f1 + f2 + f3) / 3
    return dp[state]


def solution():
    N = int(input())
    xs = list(map(int, input().split()))
    ks = [0] * 16
    for x in xs:
        ks[x] += 1
    dp = {}
    answer = f(0, ks[0], ks[1], ks[2], ks, 15, dp)
    print(answer)


solution()

`-` 내 코드의 실행 시간은 $10\text{ms}$로 파이썬 중 $1$등이다

`-` 에디토리얼은 비트마스크를 이용한 $O\left(L2^L\right)$ 풀이인데 나는 그리디하게 처리를 해서 알고리즘의 시간 복잡도를 $O(8L)$로 단축시켰다 ($L$은 가능한 좌표 개수로 여기선 $16$)

`-` 비슷한 유형의 문제: [Sushi](https://atcoder.jp/contests/dp/tasks/dp_j)

## 문제 푼 순서

|No.|문제|날짜|
|:-:|:-:|:-:|
|1  |A  |2026-10-02|
|2  |B  |2026-10-02|
|3  |E  |2026-10-03|
|4  |D  |2026-10-03|
|5  |C  |2026-10-05|
|6  |F  |2026-10-05|
|7  |H  |2026-10-05|
|8  |J  |2026-10-06|